**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 3 (MCMC Mechanics)](03_mcmc_mechanics_from_scratch.ipynb) | **Sheet 4 of 4: Production Scale**

---

# Sheet 4: Production MCMC: Multi-Chain Diagnostics, HMC & Model Comparison

This final notebook explores production Bayesian workflows: running **multiple independent Markov chains**, diagnosing convergence via **Gelman-Rubin $\hat{R}$** and **Effective Sample Size ($ESS$)**, and comparing models using **WAIC / LOO-CV**.

## Part 1: Multi-Chain MCMC Engine (`coda`)

In [ ]:
suppressPackageStartupMessages({
  library(coda)
  library(loo)
})

set.seed(42)
heights <- rnorm(50, mean = 172.5, sd = 8.0)

log_posterior <- function(mu, sigma) {
  if (sigma <= 0 || sigma >= 30) return(-Inf)
  sum(dnorm(heights, mean = mu, sd = sigma, log = TRUE)) + 
    dnorm(mu, mean = 170, sd = 15, log = TRUE) + dunif(sigma, 0, 30, log = TRUE)
}

run_single_coda_chain <- function(start_mu, start_sigma, n_iter = 5000, warmup = 1000, step_size = 0.35) {
  chain <- matrix(NA, nrow = n_iter, ncol = 2)
  colnames(chain) <- c("mu", "sigma")
  mu <- start_mu; sigma <- start_sigma
  curr_post <- log_posterior(mu, sigma)
  for (t in 1:n_iter) {
    p_mu <- rnorm(1, mu, step_size); p_sigma <- rnorm(1, sigma, step_size)
    p_post <- log_posterior(p_mu, p_sigma)
    if (log(runif(1)) < (p_post - curr_post)) {
      mu <- p_mu; sigma <- p_sigma; curr_post <- p_post
    }
    chain[t, ] <- c(mu, sigma)
  }
  return(mcmc(chain[(warmup + 1):n_iter, ]))
}

c1 <- run_single_coda_chain(150, 15)
c2 <- run_single_coda_chain(190, 5)
c3 <- run_single_coda_chain(160, 25)
c4 <- run_single_coda_chain(180, 10)
chains <- mcmc.list(c1, c2, c3, c4)

summary(chains)

## Part 2: Convergence Diagnostics (Traceplots, $\hat{R}$, $ESS$)

In [ ]:
plot(chains, col = c("navy", "darkred", "darkgreen", "purple"))

cat("=== Gelman-Rubin R-hat Diagnostic ===\n")
print(gelman.diag(chains))

cat("\n=== Effective Sample Size (ESS) ===\n")
print(effectiveSize(chains))

## Part 3: Model Comparison & Information Criteria (WAIC & LOO-CV)

How do Bayesians compare competing models? We compute **out-of-sample predictive accuracy** using **WAIC** (Widely Applicable Information Criterion) or **PSIS-LOO** (Pareto Smoothed Importance Sampling Cross-Validation):
$$\text{PSIS-LOO} = -2 \sum_{i=1}^n \log P(y_i \mid y_{-i})$$

Lower LOO values indicate superior expected predictive performance on future unseen data.

In [ ]:
# Compute pointwise log-likelihood matrix: [draws x observations]
pooled_samples <- as.matrix(chains)
n_draws <- nrow(pooled_samples)
n_obs   <- length(heights)

log_lik_matrix <- matrix(NA, nrow = n_draws, ncol = n_obs)
for (i in 1:n_obs) {
  log_lik_matrix[, i] <- dnorm(heights[i], 
                               mean = pooled_samples[, "mu"], 
                               sd   = pooled_samples[, "sigma"], 
                               log  = TRUE)
}

# Compute PSIS-LOO using R's loo library
loo_result <- loo(log_lik_matrix)
print(loo_result)
plot(loo_result, main = "PSIS-LOO Diagnostic: Pareto k values (All k < 0.5 is ideal)")

## Part 4: Hands-On Challenge Exercises

### Exercise 1: Detecting Non-Convergence with $\hat{R}$
If you run 2 chains with `n_iter = 50` without warmup, what happens to `gelman.diag(chains)`? *(Observe $\hat{R} > 1.10$, proving how $\hat{R}$ flags unfinished chains before they can mislead you).* 

### Exercise 2: Pareto $k$ Diagnostic in LOO
Look at the Pareto $k$ plot above. If one point had $k > 0.7$, what would that indicate about that specific observation? *(Hint: It indicates an influential outlier or highly leveraged data point that heavily distorts the posterior).* 

---

**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 3 (MCMC Mechanics)](03_mcmc_mechanics_from_scratch.ipynb) | **Course Complete 🎉**